# Explore choice assay data
This file re-loads the aggregated data generated in step1.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from expidite_rpi.core import configuration as root_cfg
from expidite_rpi.core.cloud_connector import CloudConnector
from datetime import timedelta

In [50]:
# Required config
AZURE_KEYS_FILE = Path.home() / ".expidite" / "keys_choiceassay.env"
assert AZURE_KEYS_FILE.exists(), f"Missing required config file: {AZURE_KEYS_FILE}"

MAX_X = 800
MAX_Y = 608

CONTAINER_NAME = "expidite-journals"
TYPE_ID = "CAPOSE"

# Local download directory (relative to notebook working directory)
DOWNLOAD_DIR = Path("./downloads")
SRC_JOURNAL_DIR = DOWNLOAD_DIR / "src_journals"
SRC_JOURNAL_DIR.mkdir(parents=True, exist_ok=True)
aggregated_csv_file = DOWNLOAD_DIR / f"aggregated_{TYPE_ID}.csv"

PREFIX = f"V3_{TYPE_ID}"
SUFFIX = ".csv"

SUBJECTS_FILE = Path("./subjects.csv")
assert SUBJECTS_FILE.exists(), f"Missing required subjects file: {SUBJECTS_FILE}"

print(f"Downloading {PREFIX} files from '{CONTAINER_NAME}' to {DOWNLOAD_DIR.resolve()}")

## Utility display functions

In [51]:
# Display methods

# Use SNS to display a heatmap of a column of interest, e.g., Tube_prob_conf by device and date.
# We calculate the date from the timestamp column.
def display_heatmap(df, column, aggfunc="mean", title="Heatmap", fmt=".1f"):

    pivot_table = df.pivot_table(index="date", columns="device_id", values=column, aggfunc=aggfunc)

    plt.figure(figsize=(12, 8))
    # Set the font to 7
    sns.heatmap(pivot_table, annot=True, fmt=fmt, cmap="YlGnBu", annot_kws={"size": 7})
    plt.title(title)
    plt.ylabel("")
    plt.xlabel("")
    plt.show()


def display_heatmap_by_hour(df, column, aggfunc="mean", title="Heatmap", fmt=".1f"):

    df["hour"] = pd.to_datetime(df["frame_start_time"], format='ISO8601').dt.floor("h")
    pivot_table = df.pivot_table(index="hour", columns="device_id", values=column, aggfunc=aggfunc)

    plt.figure(figsize=(12, 8))
    # Set the font to 7
    sns.heatmap(pivot_table, annot=True, fmt=fmt, cmap="YlGnBu", annot_kws={"size": 7})
    plt.title(title)
    plt.ylabel("")
    plt.xlabel("")
    plt.show()


def image_plot(df, x, y, title="Image Plot"):
    # Plot the L tube and R tube positions as a scatter plot, colored by device_name
    plt.figure(figsize=(12, 8))
    sns.scatterplot(
        data=df,
        x=x,
        y=y,
        hue="device_id",
        palette="tab10",
        # Use small markers with some transparency to avoid overplotting
        s=10,
        alpha=0.5,
    )
    plt.title(title)
    plt.xlabel(x)
    plt.ylabel(y)
    plt.ylim(0, MAX_Y)
    plt.xlim(0, MAX_X)
    # Since this is image data, we plot the y axes in reverse order to match the image coordinates
    plt.gca().invert_yaxis()
    # Put the legend outside the plot area to the right
    plt.legend(title="Device ID", bbox_to_anchor=(1.05, 1), loc="upper left")
    plt.show()

# Reload data from file

In [ ]:
# Load the aggregated CSV for further processing
df = pd.read_csv(aggregated_csv_file)

## Data checks

### Investigate devices where <2 tubes are detected

In [ ]:
# Tubes count can be 0, 1 or 2. Tabulate num_tubes_detected by device_id.
df.groupby("device_id")["num_tubes_detected"].value_counts()


In [ ]:
image_plot(df, "L_tube_x", "L_tube_y", title="Scatter Plot of L Tube positions by Device")
image_plot(df, "R_tube_x", "R_tube_y", title="Scatter Plot of R Tube positions by Device")

### Identify tubes that are not well positioned

If the tube X or y is within 10% of the edge of the image, that's bad

In [ ]:
BAD_MIN_X = MAX_X / 10
BAD_MAX_X = MAX_X - BAD_MIN_X
BAD_MIN_Y = MAX_Y / 10
BAD_MAX_Y = MAX_Y - BAD_MIN_Y
df["bad_tube_location"] = (
    (df["L_tube_x"] < BAD_MIN_X) |
    (df["L_tube_x"] > BAD_MAX_X) |
    (df["L_tube_y"] < BAD_MIN_Y) |
    (df["L_tube_y"] > BAD_MAX_Y) |
    (df["R_tube_x"] < BAD_MIN_X) |
    (df["R_tube_x"] > BAD_MAX_X) |
    (df["R_tube_y"] < BAD_MIN_Y) |
    (df["R_tube_y"] > BAD_MAX_Y)
)

# Tabulate all examples of bad tube locations by device_id
bad_tube_locations_df = df[df["bad_tube_location"] == True]
print(f"Total rows with bad tube locations: {len(bad_tube_locations_df)}")
bad_by_device = bad_tube_locations_df.groupby(["device_id", "date"])["bad_tube_location"].count()
print("Bad tube locations by device_id:")
print(bad_by_device)

## Reliability of tube_prob_ data

In [ ]:
# Plot the XY of the Tube_prob as a scatter plot, colored by device_name
image_plot(df, "Tube_prob_x", "Tube_prob_y", title="Scatter Plot of proboscis positions by Device")

### Reliability of tube_prob_x|y reliative to tube

In [ ]:
# Create a heatmap of the density of points in the "tube_prob_to_tube_y_distance" and "tube_prob_to_tube_x_distance" columns
# We only want instances where tp_feeding is True, since we only care about the proboscis being detected in the tube.
df_feeding = df[df["tp_feeding"] == True]

# Create cubinned heatmap of the density of points in the "tube_prob_to_tube_y_distance" and "tube_prob_to_tube_x_distance" columns
plt.figure(figsize=(8, 6))
plt.hexbin(df_feeding["tube_prob_to_tube_x_distance"],
           df_feeding["tube_prob_to_tube_y_distance"],
           gridsize=100,
           # Use a yellow to red colormap, with blue being low density and red being high density
           cmap='YlOrRd',
           bins='log')
plt.colorbar(label='log10(N)')
plt.ylim(plt.ylim()[::-1])
plt.xlabel("tube_prob_to_tube_x_distance")
plt.ylabel("tube_prob_to_tube_y_distance")
plt.show()

df_not_feeding = df[df["tp_feeding"] == False]

# Create cubinned heatmap of the density of points in the "tube_prob_to_tube_y_distance" and "tube_prob_to_tube_x_distance" columns
plt.figure(figsize=(8, 6))
plt.hexbin(df_not_feeding["tube_prob_to_tube_x_distance"],
           df_not_feeding["tube_prob_to_tube_y_distance"],
           gridsize=100,
           # Use a yellow to red colormap, with blue being low density and red being high density
           cmap='YlOrRd',
           bins='log')
plt.colorbar(label='log10(N)')
plt.ylim(plt.ylim()[::-1])
plt.xlabel("tube_prob_to_tube_x_distance")
plt.ylabel("tube_prob_to_tube_y_distance")
plt.show()

# Actual feeding data

We're going to define a feeding detection as: 
- tube_prob_conf > 0.5  AND
- tube_conf_x within +/- 150 of feeding_tube_x
- tube_conf_y within +/- 100 of feeding_tube_y

In [ ]:
# Understand what we're excluding
pre_filtered_size = len(df)
pre_filtered_high_conf_size = len(df[df["Tube_prob_conf"] >= 0.5])

df["outside_roi"] = ~(
    (df["tube_prob_to_tube_y_distance"].abs() < 150) &
    (df["tube_prob_to_tube_x_distance"].abs() < 100)
)

# Tabulate rows excluded due to bad tube locations
bad_tube_location_df = df[df["bad_tube_location"]]
bad_by_device = bad_tube_location_df.groupby(["device_id", "date"])["bad_tube_location"].count()

# Tabulate the number of rows that are excluded due to being outside the tube area or having very low confidence
table_df = df.groupby(["outside_roi", "bad_tube_location"]).size().reset_index(name="count")

# We don't actually filter out the bad_tube_location rows because we use them later in aggregate to decide whether to filter out a replicate
filtered_df = df[~df["outside_roi"]]

print(f"Filtered out {len(bad_tube_location_df)/pre_filtered_size:.1%} due to bad tube locations")
print(f"Filtered out {(df['outside_roi'].sum()/pre_filtered_size):.1%} due to ROI (tube_prob_x|y >100 pixels from the tube)")
print(f" - filtered out {(pre_filtered_high_conf_size - len(filtered_df[filtered_df['Tube_prob_conf'] >= 0.5]))/pre_filtered_high_conf_size:.1%} of high confidence rows due to ROI")

filtered_df["feeding"] = filtered_df["Tube_prob_conf"] > 0.5
print(f"Feeding detected in {filtered_df['feeding'].sum()} rows (tube_prob_conf > 0.5) \n")

print("Breakdown of rows excluded:")
print(table_df)


### Explore alternative measures of feeding...

In [ ]:
# Create a new df that defines all the seconds in the data, and whether or not there was a feeding detection in that second.
filtered_df["frame_start_time"] = pd.to_datetime(filtered_df["frame_start_time"], format='ISO8601')
filtered_df["timestamp_s"] = filtered_df["frame_start_time"].dt.floor("s")
seconds_df = filtered_df.groupby(["timestamp_s", "device_id"]).agg(
    feeding_prob_max=("Tube_prob_conf", "max"),
    feeding_prob_sum=("Tube_prob_conf", "sum"),
    feeding_x_delta_mean=("tube_prob_to_tube_x_distance", "mean"),
    feeding_y_delta_mean=("tube_prob_to_tube_y_distance", "mean"),
    frame_start_time=("timestamp_s", "first"),
).reset_index()

# Rather than using a simple "any detection > 0.5" we look at the sum of the confidence values
# for all detections in that second. If the sum is greater than 1.0, we consider that a feeding detection.
seconds_df["feeding_detected_max"] = seconds_df["feeding_prob_max"] > 0.5
seconds_df["feeding_detected_sum"] = seconds_df["feeding_prob_sum"] > 1.5
print(f"Total seconds in data: {len(seconds_df)}")
print(f"Feeding detections [conf_max]: {seconds_df['feeding_detected_max'].sum()}")
print(f"Feeding detections [conf_sum]: {seconds_df['feeding_detected_sum'].sum()}")

# Use feeding_detected_sum to define feeding clusters, since it is more robust to noise and false positives.
seconds_df["feeding_detected"] = seconds_df["feeding_detected_sum"]

# Tabulate feeding_sum by value
seconds_df["feeding_detected_sum"].value_counts().sort_index()

In [ ]:
# We want to understand how random or consistent feeding is, so we want to know if the feeding detections are clustered
# together in time, or if they are randomly distributed.
# Calculate the probability that the next second has a feeding detection given that the current second has a feeding detection.
# We can just use shift because we don't necessarily have a row for every second, only when we have events,
# so we need to check the actual timestamp_s values to see if they are consecutive seconds.
print(f"Feeding detected in seconds: {seconds_df['feeding_detected'].sum()}")
feeding_seconds_df = seconds_df[seconds_df["feeding_detected"] == True].copy()
feeding_seconds_df = feeding_seconds_df.sort_values(by=["device_id", "timestamp_s"])

# Get the next second's timestamp_s for each row, and check if it is equal to the current row's timestamp_s + 1 second.
feeding_seconds_df["next_feeding_second"] = feeding_seconds_df["timestamp_s"].shift(-1)
feeding_seconds_df["timestamp_s_plus_1"] = feeding_seconds_df["timestamp_s"] + timedelta(seconds=1)
feeding_seconds_df["consecutive"] = (
    (feeding_seconds_df["next_feeding_second"] == feeding_seconds_df["timestamp_s_plus_1"]) &
    (feeding_seconds_df["device_id"] == feeding_seconds_df["device_id"].shift(-1))
)
print(f"Feeding detected in consecutive seconds: {feeding_seconds_df['consecutive'].sum()}")

feeding_seconds_df["start_of_cluster"] = ~feeding_seconds_df["consecutive"].shift(1, fill_value=False)
feeding_seconds_df["end_of_cluster"] = ~feeding_seconds_df["consecutive"]
feeding_seconds_df["cluster_id"] = feeding_seconds_df["start_of_cluster"].cumsum()

feeding_clusters_df = feeding_seconds_df.groupby("cluster_id").agg(
    frame_start_time=("timestamp_s", "min"),
    end_time=("timestamp_s", "max"),
    duration=("timestamp_s", lambda x: (x.max() - x.min()).total_seconds() + 1),
    device_id=("device_id", "first"),
).reset_index()

print(f"Total feeding clusters: {len(feeding_clusters_df)}")
print(f"Feeding clusters with duration > 1 second: {len(feeding_clusters_df[feeding_clusters_df['duration'] > 1])}")

# Create a survivorship plot of the feeding cluster durations
plt.figure(figsize=(8, 6))
sns.ecdfplot(feeding_clusters_df["duration"])
plt.title("Survivorship Plot of Feeding Cluster Durations")
# Make the x axis a log scale to better visualize the long tail of the distribution
plt.xscale("log")
plt.xlabel("Duration (seconds)")
plt.ylabel("Proportion of Clusters")
plt.show()


In [ ]:
# Repeat, but allow up to a 5 second gap in the feeding detection
gap_size = 5

print(f"Feeding detected in seconds: {seconds_df['feeding_detected'].sum()}")
feeding_seconds_df = seconds_df[seconds_df["feeding_detected"] == True].copy()
feeding_seconds_df = feeding_seconds_df.sort_values(by=["device_id", "timestamp_s"])

# Get the next second's timestamp_s for each row, and check if it is equal to the current row's timestamp_s + 1 second.
feeding_seconds_df["next_feeding_second"] = feeding_seconds_df["timestamp_s"].shift(-1)
feeding_seconds_df["timestamp_s_plus_gap"] = feeding_seconds_df["timestamp_s"] + timedelta(seconds=gap_size+1)
feeding_seconds_df["consecutive"] = (
    (feeding_seconds_df["next_feeding_second"] <= feeding_seconds_df["timestamp_s_plus_gap"]) &
    (feeding_seconds_df["device_id"] == feeding_seconds_df["device_id"].shift(-1))
)
print(f"Feeding detected in consecutive seconds: {feeding_seconds_df['consecutive'].sum()}")

feeding_seconds_df["start_of_cluster"] = ~feeding_seconds_df["consecutive"].shift(1, fill_value=False)
feeding_seconds_df["end_of_cluster"] = ~feeding_seconds_df["consecutive"]
feeding_seconds_df["cluster_id"] = feeding_seconds_df["start_of_cluster"].cumsum()

feeding_clusters_df = feeding_seconds_df.groupby("cluster_id").agg(
    start_time=("timestamp_s", "min"),
    end_time=("timestamp_s", "max"),
    duration=("timestamp_s", lambda x: (x.max() - x.min()).total_seconds() + 1),
    device_id=("device_id", "first"),
).reset_index()

print(f"Total feeding clusters: {len(feeding_clusters_df)}")
print(f"Feeding clusters with duration > 1 second: {len(feeding_clusters_df[feeding_clusters_df['duration'] > 1])}")

# Create a survivorship plot of the feeding cluster durations
plt.figure(figsize=(8, 6))
sns.ecdfplot(feeding_clusters_df["duration"])
plt.title("Survivorship Plot of Feeding Cluster Durations (up to 5 second gap)")
plt.xlabel("Duration (seconds)")
plt.ylabel("Proportion of Clusters")
plt.xscale("log")
plt.show()


In [ ]:
# Plot a histogram of the feeding cluster durations
plt.figure(figsize=(8, 6))
sns.histplot(feeding_clusters_df["duration"], bins=50, kde=True)
plt.title("Distribution of Feeding Cluster Durations")
plt.xlabel("Duration (seconds)")
plt.ylabel("Frequency")
plt.show()

In [ ]:
# Display a heatmap of the total feeding frames by device and date
display_heatmap(filtered_df, "feeding", "sum", "Total Feeding Frames by Device")
#display_heatmap_by_hour(filtered_df, "feeding", "sum", "Total Feeding Frames by Device and Hour")
#display_heatmap_by_hour(feeding_seconds_df, "feeding_detected", "sum", "Total Feeding Seconds by Device")

# Feeding data by treatment / replicate

Aggregate data to give a total feeding for each experimental replicate
Replicates are uniquely defined by Round + RPi 

Exclude replciates where:
- subjects.csv exclude = 1
- bad_tube_location count > 300

In [ ]:
replicate_df = filtered_df.groupby(["RPi", "Round", "Tube"]).agg(
    total_feeding_frames=("feeding", "sum"),
    # Get first of Date,RPi version,Colony,Treatment,Tube,Solution,Change
    rpi_number=("rpi_number", "first"),
    date=("Date", "first"),
    rpi_version=("RPi version", "first"),
    colony=("Colony", "first"),
    treatment=("Treatment", "first"),
    tube=("Tube", "first"),
    solution=("Solution", "first"),
    change=("Change", "first"),
    exclude=("Exclude", "first"),
    bad_tube_location=("bad_tube_location", "sum")
).reset_index()
orig_len = len(replicate_df)
print(f"{len(filtered_df)} rows aggregated into {orig_len} replicates by RPi and Round")

# Filter out excluded replicates
# Some rows in subjects.csv have Exclude=1 where a tube leaked
replicate_df = replicate_df[replicate_df["exclude"] != 1]
print(f"Filtered out {orig_len - len(replicate_df)} replicates due to Exclude=1 in subjects.csv")

# Exclude replicates where bad_tube_location is >300
orig_len = len(replicate_df)
replicate_df = replicate_df[replicate_df["bad_tube_location"] <= 300]
print(f"Filtered out {orig_len - len(replicate_df)} replicates due to bad_tube_location > 300")

In [ ]:
# Scatter plot total_feeding_frames against change, colored by treatment; with a regression line for each treatment
plt.figure(figsize=(8, 6))
sns.lmplot(data=replicate_df, x="change", y="total_feeding_frames", hue="solution", height=6, aspect=8/6, scatter_kws={"s":100})
plt.title("Total Feeding Frames by Change, Colored by Solution")
plt.xlabel("Change")
plt.ylabel("Total Feeding Frames")
plt.show()

In [ ]:
# Plot boxplots of total_feeding_frames, colored by Treatment
plt.figure(figsize=(8, 6))
sns.boxplot(data=replicate_df, x="solution", y="total_feeding_frames")
plt.title("Total Feeding Frames by solution")
plt.xlabel("solution")
plt.ylabel("Total Feeding Frames")
plt.show()

plt.figure(figsize=(8, 6))
sns.boxplot(data=replicate_df, x="solution", y="change")
plt.title("Change by solution")
plt.xlabel("solution")
plt.ylabel("Change")
plt.show()